# Crack Segmentation — Colab 訓練筆記

**使用前請先確認：**
1. 右上角 Runtime → Change runtime type → **T4 GPU**（免費）
2. 依序執行每個 Cell

## Cell 1：確認 GPU

In [ ]:
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 2：掛載 Google Drive（儲存模型用）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/crack_SEG'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive 目錄已就緒:', DRIVE_DIR)

## Cell 3：上傳專案 zip 並解壓縮

**執行前準備：** 先把整個 `crack_SEG` 資料夾壓成 zip（**不含 dataset/**），檔名建議 `crack_SEG.zip`。

執行後會跳出「選擇檔案」視窗，選你剛壓好的 zip 上傳。

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # 選擇 crack_SEG.zip
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# 找出解壓後的資料夾名稱
import glob
candidates = [d for d in glob.glob('/content/crack*') if os.path.isdir(d)]
PROJECT_DIR = candidates[0] if candidates else '/content/crack_SEG'
print('專案目錄:', PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('目前路徑:', os.getcwd())

## Cell 4：安裝相依套件

In [ ]:
# Colab 預裝 TF 2.x，只需補裝這幾個
!pip install -q imgaug==0.4.0
!pip install -q "numpy==1.24.0" --force-reinstall
!pip install -q tqdm scikit-learn
print('套件安裝完成')

## Cell 5：從 Kaggle 下載資料集

**執行前準備：** 去 [kaggle.com/settings](https://www.kaggle.com/settings) → API → **Create New Token** → 下載 `kaggle.json`。

執行後選擇該 `kaggle.json` 上傳。

In [ ]:
from google.colab import files
import os

print('請上傳 kaggle.json ...')
uploaded = files.upload()  # 選擇 kaggle.json

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json 設定完成')

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d lakshaymiddha/crack-segmentation-dataset -p /content/dataset_zip

import zipfile, pathlib
os.makedirs('dataset/image', exist_ok=True)
os.makedirs('dataset/mask', exist_ok=True)

with zipfile.ZipFile('/content/dataset_zip/crack-segmentation-dataset.zip', 'r') as z:
    names = z.namelist()
    img_files = [n for n in names if n.startswith('crack_segmentation_dataset/images/') and not n.endswith('/')]
    msk_files = [n for n in names if n.startswith('crack_segmentation_dataset/masks/') and not n.endswith('/')]
    for src in img_files:
        fname = pathlib.Path(src).name
        with z.open(src) as f_in, open(f'dataset/image/{fname}', 'wb') as f_out:
            f_out.write(f_in.read())
    for src in msk_files:
        fname = pathlib.Path(src).name
        with z.open(src) as f_in, open(f'dataset/mask/{fname}', 'wb') as f_out:
            f_out.write(f_in.read())

print('image:', len(list(pathlib.Path('dataset/image').iterdir())))
print('mask: ', len(list(pathlib.Path('dataset/mask').iterdir())))

# 清理暫存，釋放空間
!rm -rf /content/dataset_zip
print('資料集就緒')

## Cell 6：開始訓練

T4 GPU 有 15GB VRAM，可以用較大的設定：
- `IMG_SIZE=448`, `BATCH_SIZE=8`，訓練品質最好
- 100 epochs 約需 **1.5～2 小時**

In [ ]:
!python train.py \
    --data_path dataset \
    --model_type unet \
    --img_size 448 \
    --batch_size 8 \
    --epochs 100 \
    --model_output checkpoints/best_model.h5 \
    --figure_dir runs

## Cell 7：儲存模型到 Google Drive（重要！Colab 關閉後檔案會消失）

In [ ]:
import shutil, os

DRIVE_DIR = '/content/drive/MyDrive/crack_SEG'
os.makedirs(DRIVE_DIR, exist_ok=True)

# 複製模型
shutil.copy('checkpoints/best_model.h5', f'{DRIVE_DIR}/best_model.h5')
print('模型已存到 Drive:', f'{DRIVE_DIR}/best_model.h5')

# 複製訓練曲線圖
shutil.copytree('runs', f'{DRIVE_DIR}/runs', dirs_exist_ok=True)
print('訓練曲線已存到 Drive:', f'{DRIVE_DIR}/runs/')

## Cell 8（可選）：執行評估，取得真實指標

In [ ]:
!python evaluate.py \
    --data_path dataset \
    --model_path checkpoints/best_model.h5 \
    --output_dir eval_results

# 也存到 Drive
shutil.copytree('eval_results', f'{DRIVE_DIR}/eval_results', dirs_exist_ok=True)
print('評估結果已存到 Drive')